# Boilerplate

In [5]:
import torch
from transformers import PeAudioVideoModel, PeAudioVideoProcessor,AutoProcessor, AutoModel
import time
import numpy as np
from pathlib import Path
import glob
import os
import re
from natsort import natsorted  
import gc  

BASE_DIR = Path("/home/amin/Research/Representation/Movie/data/Setareh/Data")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device is {DEVICE}")

device is cuda


# loop

In [6]:
# --- CONFIG ---
SEGMENT_DURATIONS = [1.0, 2.0, 5.0, 10.0]  
# ----------------

def extract_part_number(filename):
    match = re.search(r'part[_-]?(\d+)', filename.stem, re.IGNORECASE)
    if match:
        return int(match.group(1))
    return None

print(f"Checking data compatibility in: {BASE_DIR}\n")

for seg_sec in SEGMENT_DURATIONS:
    all_videos = []
    all_audios = []
    
    # Optional: Logic for sliding window steps could go here
    # steps = [seg_sec] 
    # for step in steps: 

    print(f"--- Checking {int(seg_sec)}s segments ---")

    for run_idx in range(1, 19):
        vid_chunk_dir = BASE_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks_{int(seg_sec)}s"
        aud_chunk_dir = BASE_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks_{int(seg_sec)}s"

        if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
            print(f"Video {run_idx}: 0 paired chunks (Dir missing)")
            continue

        vids = natsorted(list(vid_chunk_dir.glob("*.mp4")))
        # Search for .wav (preferred) or fallback to .m4a if needed
        auds = natsorted(list(aud_chunk_dir.glob("*.wav")))
        if not auds: 
             auds = natsorted(list(aud_chunk_dir.glob("*.m4a")))

        vid_parts = {extract_part_number(v): v for v in vids if extract_part_number(v) is not None}
        aud_parts = {extract_part_number(a): a for a in auds if extract_part_number(a) is not None}

        common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))

        for part_num in common_parts:
            all_videos.append(vid_parts[part_num])
            all_audios.append(aud_parts[part_num])

        if len(vids) == len(auds):    
            print(f"Video {run_idx}: {len(common_parts)} paired chunks (video: {len(vids)}, audio: {len(auds)})")
        else:
            raise RuntimeError("ERROR! mismatched segment length")

    paired = list(zip(all_videos, all_audios))
    print(f"{'='*60}")
    print(f"Total: {len(paired)} paired {int(seg_sec)}s segments")
    print(f"{'='*60}\n")

Checking data compatibility in: /home/amin/Research/Representation/Movie/data/Setareh/Data

--- Checking 1s segments ---
Video 1: 212 paired chunks (video: 212, audio: 212)
Video 2: 220 paired chunks (video: 220, audio: 220)
Video 3: 186 paired chunks (video: 186, audio: 186)
Video 4: 62 paired chunks (video: 62, audio: 62)
Video 5: 82 paired chunks (video: 82, audio: 82)
Video 6: 226 paired chunks (video: 226, audio: 226)
Video 7: 258 paired chunks (video: 258, audio: 258)
Video 8: 248 paired chunks (video: 248, audio: 248)
Video 9: 82 paired chunks (video: 82, audio: 82)
Video 10: 180 paired chunks (video: 180, audio: 180)
Video 11: 184 paired chunks (video: 184, audio: 184)
Video 12: 202 paired chunks (video: 202, audio: 202)
Video 13: 140 paired chunks (video: 140, audio: 140)
Video 14: 82 paired chunks (video: 82, audio: 82)
Video 15: 232 paired chunks (video: 232, audio: 232)
Video 16: 230 paired chunks (video: 230, audio: 230)
Video 17: 254 paired chunks (video: 254, audio: 254)

In [ ]:
import torch
import numpy as np
import os
import re
from pathlib import Path
from natsort import natsorted


model_id_list = ["facebook/pe-av-small","facebook/pe-av-base","facebook/pe-av-large", "facebook/pe-av-base-16-frame", "facebook/pe-av-large-16-frame"]

for model_id in model_id_list:
    os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"  # 5 minutes

    model = PeAudioVideoModel.from_pretrained(model_id).to(DEVICE)
    processor = PeAudioVideoProcessor.from_pretrained(model_id)
    print(f"{model_id} & Processor Ready")

    # --- CONFIG ---
    OUTPUT_BASE = Path(f"/home/amin/Research/Representation/Movie/outputs/model_embeddings/{model_id.split("/")[-1]}")
    BATCH_SIZE = 8
    SEGMENT_DURATIONS = [1.0, 2.0, 5.0, 10.0] 
    # ----------------

    for seg_sec in SEGMENT_DURATIONS:
        
        # 1. Gather Pairs (Re-running logic to ensure safety per loop iteration)
        all_videos = []
        all_audios = []
        
        for run_idx in range(1, 19):
            vid_chunk_dir = BASE_DIR / f"Video{run_idx}" / f"Video{run_idx}_chunks_{int(seg_sec)}s"
            aud_chunk_dir = BASE_DIR / f"Audio{run_idx}" / f"Audio{run_idx}_chunks_{int(seg_sec)}s"
            
            if not vid_chunk_dir.exists() or not aud_chunk_dir.exists():
                continue

            vids = natsorted(list(vid_chunk_dir.glob("*.mp4")))
            auds = natsorted(list(aud_chunk_dir.glob("*.wav")))
            if not auds: auds = natsorted(list(aud_chunk_dir.glob("*.m4a")))

            vid_parts = {extract_part_number(v): v for v in vids if extract_part_number(v) is not None}
            aud_parts = {extract_part_number(a): a for a in auds if extract_part_number(a) is not None}
            common_parts = sorted(set(vid_parts.keys()) & set(aud_parts.keys()))

            for part_num in common_parts:
                all_videos.append(vid_parts[part_num])
                all_audios.append(aud_parts[part_num])
            if len(vids) == len(auds):    
                print(f"Video {run_idx}: {len(common_parts)} paired chunks (video: {len(vids)}, audio: {len(auds)})")
            else:
                raise RuntimeError("ERROR! mismatched segment length")

        paired = list(zip(all_videos, all_audios))
        print(f"{'='*60}")
        print(f"Total: {len(paired)} paired {int(seg_sec)}s segments")
        print(f"{'='*60}")

        if len(paired) == 0:
            continue

        # 2. Extract
        OUTPUT_PATH = OUTPUT_BASE / f"{int(seg_sec)}s"
        os.makedirs(OUTPUT_PATH, exist_ok=True)
        
        video_out, audio_out, av_out = [], [], []

        with torch.inference_mode():
            for i in range(0, len(paired), BATCH_SIZE):
                batch_pairs = paired[i:i + BATCH_SIZE]
                v_batch = [str(v) for v, a in batch_pairs]
                a_batch = [str(a) for v, a in batch_pairs]
                
                inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(torch.float32).to(DEVICE)
                
                with torch.autocast(device_type="cuda", dtype=torch.float32):
                    outputs = model(**inputs)
                
                video_out.append(outputs.video_embeds.float().cpu())
                audio_out.append(outputs.audio_embeds.float().cpu())
                av_out.append(outputs.audio_video_embeds.float().cpu())
                
                if (i // BATCH_SIZE) % 50 == 0 or i == 0:
                    print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

        # 3. Save
        video_embeds = torch.cat(video_out, dim=0)
        audio_embeds = torch.cat(audio_out, dim=0)
        av_embeds = torch.cat(av_out, dim=0)

        print(f"Done!")
        print(f"Video embeddings: {video_embeds.shape}")
        print(f"Audio embeddings: {audio_embeds.shape}")
        print(f"AV embeddings:    {av_embeds.shape}")

        np.save(OUTPUT_PATH / 'peav_small_16-frame_av.npy', av_embeds.numpy())
        np.save(OUTPUT_PATH / 'peav_small_16-frame_a.npy', audio_embeds.numpy())
        np.save(OUTPUT_PATH / 'peav_small_16-frame_v.npy', video_embeds.numpy())
        print(f"embedding npy files saved to {OUTPUT_PATH}\n")
        del av_embeds,audio_embeds, video_embeds
    del model
    gc.collect()

# Misc

## PE-AV-Large

In [ ]:
# Base paths
MODEL_ID = 'facebook/pe-av-large'
MODEL_DIR = Path("/home/amin/.cache/huggingface/pe-av-large") 
LOCAL_FILES_ONLY = False
OUTPUT_PATH = Path("/home/amin/amin/outputs/model_embeddings/pe-av-large")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
LOCAL_FILES_ONLY = False
MAX_RUNS = 18
BATCH_SIZE = 8

In [ ]:
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
print(f"Loading {MODEL_ID}...")


if LOCAL_FILES_ONLY:
    model = PeAudioVideoModel.from_pretrained(MODEL_DIR, local_files_only=True).to(DEVICE)
    processor = PeAudioVideoProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
else:
    model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(DEVICE)
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Model & Processor Ready")

Loading facebook/pe-av-large...


In [ ]:
# Extract embeddings
video_out, audio_out, av_out = [], [], []
clip_names = []
vid_ids = []

with torch.inference_mode():
    for i in range(0, len(paired), BATCH_SIZE):
        batch_pairs = paired[i:i + BATCH_SIZE]
        v_batch = [str(v) for v, a in batch_pairs]
        a_batch = [str(a) for v, a in batch_pairs]
        
        inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(DEVICE)
        
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**inputs)
        
        video_out.append(outputs.video_embeds.float().cpu())
        audio_out.append(outputs.audio_embeds.float().cpu())
        av_out.append(outputs.audio_video_embeds.float().cpu())
        
        for v, a in batch_pairs:
            clip_names.append(v.stem)
            vid_ids.append(v.parent.parent.name)
        
        if (i // BATCH_SIZE) % 50 == 0 or i == 0:
            print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

video_embeds = torch.cat(video_out, dim=0)
audio_embeds = torch.cat(audio_out, dim=0)
av_embeds = torch.cat(av_out, dim=0)

print(f"\nDone!")
print(f"Video embeddings: {video_embeds.shape}")
print(f"Audio embeddings: {audio_embeds.shape}")
print(f"AV embeddings:    {av_embeds.shape}")

In [ ]:
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_av.npy'),av_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_a.npy'),audio_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_v.npy'),video_embeds.numpy())
print(f"joint embedding npy files saved to {OUTPUT_PATH}")

In [ ]:
np.savez(
    OUTPUT_PATH.with_suffix('.npz'),
    video=video_embeds.numpy(),
    audio=audio_embeds.numpy(),
    av=av_embeds.numpy(),
    clip_names=np.array(clip_names),
    vid_ids=np.array(vid_ids),
)
print(f"joint modality .npz saved to {OUTPUT_PATH.with_suffix('.npz')}")

In [ ]:
del model, processor  
del video_out, audio_out, av_out  
del video_embeds, audio_embeds, av_embeds  
torch.cuda.empty_cache()  
gc.collect()  

## PE-AV-base

In [ ]:
# Base paths
MODEL_ID = 'facebook/pe-av-base'
MODEL_DIR = Path("/home/amin/.cache/huggingface/pe-av-base") 
LOCAL_FILES_ONLY = False
OUTPUT_PATH = Path("/home/amin/amin/outputs/model_embeddings/pe-av-base")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
LOCAL_FILES_ONLY = False
MAX_RUNS = 18
BATCH_SIZE = 8

In [ ]:
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
print(f"Loading {MODEL_ID}...")


if LOCAL_FILES_ONLY:
    model = PeAudioVideoModel.from_pretrained(MODEL_DIR, local_files_only=True).to(DEVICE)
    processor = PeAudioVideoProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
else:
    model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(DEVICE)
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Model & Processor Ready")

## Extract embeddings

In [ ]:
# Extract embeddings
video_out, audio_out, av_out = [], [], []
clip_names = []
vid_ids = []

with torch.inference_mode():
    for i in range(0, len(paired), BATCH_SIZE):
        batch_pairs = paired[i:i + BATCH_SIZE]
        v_batch = [str(v) for v, a in batch_pairs]
        a_batch = [str(a) for v, a in batch_pairs]
        
        inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(DEVICE)
        
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**inputs)
        
        video_out.append(outputs.video_embeds.float().cpu())
        audio_out.append(outputs.audio_embeds.float().cpu())
        av_out.append(outputs.audio_video_embeds.float().cpu())
        
        for v, a in batch_pairs:
            clip_names.append(v.stem)
            vid_ids.append(v.parent.parent.name)
        
        if (i // BATCH_SIZE) % 50 == 0 or i == 0:
            print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

video_embeds = torch.cat(video_out, dim=0)
audio_embeds = torch.cat(audio_out, dim=0)
av_embeds = torch.cat(av_out, dim=0)

print(f"\nDone!")
print(f"Video embeddings: {video_embeds.shape}")
print(f"Audio embeddings: {audio_embeds.shape}")
print(f"AV embeddings:    {av_embeds.shape}")

In [ ]:
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_av.npy'),av_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_a.npy'),audio_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_v.npy'),video_embeds.numpy())
print(f"joint embedding npy files saved to {OUTPUT_PATH}")

In [ ]:
import numpy as np

np.savez(
    OUTPUT_PATH.with_suffix('.npz'),
    video=video_embeds.numpy(),
    audio=audio_embeds.numpy(),
    av=av_embeds.numpy(),
    clip_names=np.array(clip_names),
    vid_ids=np.array(vid_ids),
)
print(f"joint modality .npz saved to {OUTPUT_PATH.with_suffix('.npz')}")

In [ ]:
del model, processor  
del video_out, audio_out, av_out  
del video_embeds, audio_embeds, av_embeds  
torch.cuda.empty_cache()  
gc.collect()  

## PE-AV-small

In [ ]:
# Base paths
MODEL_ID = 'facebook/pe-av-small'
MODEL_DIR = Path("/home/amin/.cache/huggingface/pe-av-small") 
LOCAL_FILES_ONLY = False
OUTPUT_PATH = Path("/home/amin/amin/outputs/model_embeddings/pe-av-small")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
LOCAL_FILES_ONLY = False
MAX_RUNS = 18
BATCH_SIZE = 8

In [ ]:
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
print(f"Loading {MODEL_ID}...")


if LOCAL_FILES_ONLY:
    model = PeAudioVideoModel.from_pretrained(MODEL_DIR, local_files_only=True).to(DEVICE)
    processor = PeAudioVideoProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
else:
    model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(DEVICE)
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Model & Processor Ready")

## Extract embeddings

In [ ]:
# Extract embeddings
video_out, audio_out, av_out = [], [], []
clip_names = []
vid_ids = []

with torch.inference_mode():
    for i in range(0, len(paired), BATCH_SIZE):
        batch_pairs = paired[i:i + BATCH_SIZE]
        v_batch = [str(v) for v, a in batch_pairs]
        a_batch = [str(a) for v, a in batch_pairs]
        
        inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(DEVICE)
        
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**inputs)
        
        video_out.append(outputs.video_embeds.float().cpu())
        audio_out.append(outputs.audio_embeds.float().cpu())
        av_out.append(outputs.audio_video_embeds.float().cpu())
        
        for v, a in batch_pairs:
            clip_names.append(v.stem)
            vid_ids.append(v.parent.parent.name)
        
        if (i // BATCH_SIZE) % 50 == 0 or i == 0:
            print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

video_embeds = torch.cat(video_out, dim=0)
audio_embeds = torch.cat(audio_out, dim=0)
av_embeds = torch.cat(av_out, dim=0)

print(f"\nDone!")
print(f"Video embeddings: {video_embeds.shape}")
print(f"Audio embeddings: {audio_embeds.shape}")
print(f"AV embeddings:    {av_embeds.shape}")

In [ ]:
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_av.npy'),av_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_a.npy'),audio_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_v.npy'),video_embeds.numpy())
print(f"joint embedding npy files saved to {OUTPUT_PATH}")

In [ ]:
import numpy as np

np.savez(
    OUTPUT_PATH.with_suffix('.npz'),
    video=video_embeds.numpy(),
    audio=audio_embeds.numpy(),
    av=av_embeds.numpy(),
    clip_names=np.array(clip_names),
    vid_ids=np.array(vid_ids),
)
print(f"joint modality .npz saved to {OUTPUT_PATH.with_suffix('.npz')}")

In [ ]:
del model, processor  
del video_out, audio_out, av_out  
del video_embeds, audio_embeds, av_embeds  
torch.cuda.empty_cache()  
gc.collect()  

## PE-AV-base-16-frame

In [ ]:
# Base paths
MODEL_ID = 'facebook/pe-av-base-16-frame'
MODEL_DIR = Path("/home/amin/.cache/huggingface/pe-av-base-16-frame") 
LOCAL_FILES_ONLY = False
OUTPUT_PATH = Path("/home/amin/amin/outputs/model_embeddings/pe-av-base-16-frame")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
LOCAL_FILES_ONLY = False
MAX_RUNS = 18
BATCH_SIZE = 8

In [ ]:
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
print(f"Loading {MODEL_ID}...")


if LOCAL_FILES_ONLY:
    model = PeAudioVideoModel.from_pretrained(MODEL_DIR, local_files_only=True).to(DEVICE)
    processor = PeAudioVideoProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
else:
    model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(DEVICE)
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Model & Processor Ready")

## Extract embeddings

In [ ]:
# Extract embeddings
video_out, audio_out, av_out = [], [], []
clip_names = []
vid_ids = []

with torch.inference_mode():
    for i in range(0, len(paired), BATCH_SIZE):
        batch_pairs = paired[i:i + BATCH_SIZE]
        v_batch = [str(v) for v, a in batch_pairs]
        a_batch = [str(a) for v, a in batch_pairs]
        
        inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(DEVICE)
        
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**inputs)
        
        video_out.append(outputs.video_embeds.float().cpu())
        audio_out.append(outputs.audio_embeds.float().cpu())
        av_out.append(outputs.audio_video_embeds.float().cpu())
        
        for v, a in batch_pairs:
            clip_names.append(v.stem)
            vid_ids.append(v.parent.parent.name)
        
        if (i // BATCH_SIZE) % 50 == 0 or i == 0:
            print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

video_embeds = torch.cat(video_out, dim=0)
audio_embeds = torch.cat(audio_out, dim=0)
av_embeds = torch.cat(av_out, dim=0)

print(f"\nDone!")
print(f"Video embeddings: {video_embeds.shape}")
print(f"Audio embeddings: {audio_embeds.shape}")
print(f"AV embeddings:    {av_embeds.shape}")

In [ ]:
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_av.npy'),av_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_a.npy'),audio_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_v.npy'),video_embeds.numpy())
print(f"joint embedding npy files saved to {OUTPUT_PATH}")

In [ ]:
import numpy as np

np.savez(
    OUTPUT_PATH.with_suffix('.npz'),
    video=video_embeds.numpy(),
    audio=audio_embeds.numpy(),
    av=av_embeds.numpy(),
    clip_names=np.array(clip_names),
    vid_ids=np.array(vid_ids),
)
print(f"joint modality .npz saved to {OUTPUT_PATH.with_suffix('.npz')}")

In [ ]:
del model, processor  
del video_out, audio_out, av_out  
del video_embeds, audio_embeds, av_embeds  
torch.cuda.empty_cache()  
gc.collect()  

## PE-AV-Large-16-frame

In [ ]:
# Base paths
MODEL_ID = 'facebook/pe-av-large-16-frame'
MODEL_DIR = Path("/home/amin/.cache/huggingface/pe-av-large-16-frame") 
LOCAL_FILES_ONLY = False
OUTPUT_PATH = Path("/home/amin/amin/outputs/model_embeddings/pe-av-large-16-frame")
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)
LOCAL_FILES_ONLY = False
MAX_RUNS = 18
BATCH_SIZE = 8

In [ ]:
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "300"
print(f"Loading {MODEL_ID}...")


if LOCAL_FILES_ONLY:
    model = PeAudioVideoModel.from_pretrained(MODEL_DIR, local_files_only=True).to(DEVICE)
    processor = PeAudioVideoProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
else:
    model = AutoModel.from_pretrained(MODEL_ID, trust_remote_code=True).to(DEVICE)
    processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)

print("Model & Processor Ready")

In [ ]:
# Extract embeddings
video_out, audio_out, av_out = [], [], []
clip_names = []
vid_ids = []

with torch.inference_mode():
    for i in range(0, len(paired), BATCH_SIZE):
        batch_pairs = paired[i:i + BATCH_SIZE]
        v_batch = [str(v) for v, a in batch_pairs]
        a_batch = [str(a) for v, a in batch_pairs]
        
        inputs = processor(videos=v_batch, audio=a_batch, return_tensors="pt").to(DEVICE)
        
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
            outputs = model(**inputs)
        
        video_out.append(outputs.video_embeds.float().cpu())
        audio_out.append(outputs.audio_embeds.float().cpu())
        av_out.append(outputs.audio_video_embeds.float().cpu())
        
        for v, a in batch_pairs:
            clip_names.append(v.stem)
            vid_ids.append(v.parent.parent.name)
        
        if (i // BATCH_SIZE) % 50 == 0 or i == 0:
            print(f"Processed {i + len(batch_pairs)}/{len(paired)}")

video_embeds = torch.cat(video_out, dim=0)
audio_embeds = torch.cat(audio_out, dim=0)
av_embeds = torch.cat(av_out, dim=0)

print(f"\nDone!")
print(f"Video embeddings: {video_embeds.shape}")
print(f"Audio embeddings: {audio_embeds.shape}")
print(f"AV embeddings:    {av_embeds.shape}")

In [ ]:
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_av.npy'),av_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_a.npy'),audio_embeds.numpy())
np.save(Path(OUTPUT_PATH / f'{MODEL_ID}_v.npy'),video_embeds.numpy())
print(f"joint embedding npy files saved to {OUTPUT_PATH}")

In [ ]:
import numpy as np

np.savez(
    OUTPUT_PATH.with_suffix('.npz'),
    video=video_embeds.numpy(),
    audio=audio_embeds.numpy(),
    av=av_embeds.numpy(),
    clip_names=np.array(clip_names),
    vid_ids=np.array(vid_ids),
)
print(f"joint modality .npz saved to {OUTPUT_PATH.with_suffix('.npz')}")

In [ ]:
del model, processor  
del video_out, audio_out, av_out  
del video_embeds, audio_embeds, av_embeds  
torch.cuda.empty_cache()  
gc.collect()  